# Capa de analisis - EDA y preprocesamiento

Dataset: NASA Exoplanet Archive + Kepler Objects of Interest.

Objetivo de esta capa: entender que hay en los datos crudos, clasificar atributos, revisar calidad de datos y dejar definido el preprocesamiento antes de modelar.

Tipos de datos usados:

- nominal
- binario
- ordinal
- numerico


In [ ]:
from __future__ import annotations

from io import StringIO
from pathlib import Path
import math

import numpy as np
import pandas as pd
import plotly.express as px
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NAVY = "#1C3257"
TERRA = "#AA4B37"
SAND = "#F4EFE6"
INK = "#1A1A1A"
PLOTLY_TEMPLATE = "plotly_white"
PLOTLY_FONT = dict(family="Helvetica, Arial, sans-serif", color=INK, size=13)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 80)


## 1. Funciones auxiliares

Estas funciones siguen el estilo de las actividades anteriores: resumen, clasificacion de atributos, nulos, outliers y limpieza basica de espacios en blanco.


In [ ]:
def clasificar_atributos(
    df: pd.DataFrame,
    hints: dict[str, str] | None = None,
    uso: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Clasifica columnas usando solo: nominal, binario, ordinal y numerico."""
    tipos_validos = {"nominal", "binario", "ordinal", "numerico"}
    hints = hints or {}
    uso = uso or {}
    filas = []

    for col in df.columns:
        s = df[col]
        n_unique = int(s.nunique(dropna=True))
        n_missing = int(s.isna().sum())
        pct_missing = round(float(s.isna().mean() * 100), 2)
        dtype = str(s.dtype)

        if col in hints:
            tipo = hints[col]
        elif pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
            tipo = "nominal"
        elif n_unique == 2:
            tipo = "binario"
        elif pd.api.types.is_integer_dtype(s) and n_unique <= 12:
            tipo = "ordinal"
        elif pd.api.types.is_numeric_dtype(s):
            tipo = "numerico"
        else:
            tipo = "nominal"

        if tipo not in tipos_validos:
            raise ValueError(f"Tipo no permitido para {col}: {tipo}")

        ejemplos = s.dropna().astype(str).unique()[:4]
        filas.append(
            {
                "columna": col,
                "dtype_pandas": dtype,
                "tipo_dato": tipo,
                "uso": uso.get(col, "analisis"),
                "n_unique": n_unique,
                "n_missing": n_missing,
                "pct_missing": pct_missing,
                "ejemplos": list(ejemplos),
            }
        )

    return pd.DataFrame(filas).sort_values(
        by=["uso", "tipo_dato", "pct_missing", "columna"],
        ascending=[True, True, False, True],
    ).reset_index(drop=True)


def limpiar_espacios_blanco(df: pd.DataFrame) -> pd.DataFrame:
    """Quita espacios al inicio/final en columnas nominales y convierte cadenas vacias en NaN."""
    limpio = df.copy()
    columnas_texto = limpio.select_dtypes(include=["object", "string"]).columns
    for col in columnas_texto:
        limpio[col] = limpio[col].astype("string").str.strip()
        limpio[col] = limpio[col].replace({"": pd.NA})
    return limpio


def info_texto(df: pd.DataFrame) -> str:
    buffer = StringIO()
    df.info(buf=buffer)
    return buffer.getvalue()


def resumen_dataframe(nombre: str, df: pd.DataFrame) -> dict:
    return {
        "dataset": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "filas_duplicadas": int(df.duplicated().sum()),
        "celdas_nulas": int(df.isna().sum().sum()),
        "pct_nulos_total": round(float(df.isna().sum().sum() / df.size * 100), 2),
        "columnas_con_nulos": int((df.isna().sum() > 0).sum()),
    }


def columnas_con_mas_nulos(df: pd.DataFrame, n: int = 15) -> pd.DataFrame:
    tabla = df.isna().agg(["sum", "mean"]).T
    tabla.columns = ["n_missing", "pct_missing"]
    tabla["pct_missing"] = (tabla["pct_missing"] * 100).round(2)
    return tabla.sort_values("pct_missing", ascending=False).head(n)


def resumen_numerico(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    columnas_validas = [col for col in columnas if col in df.columns]
    return df[columnas_validas].describe().T.round(3)


def detectar_outliers_iqr(s: pd.Series, k: float = 1.5) -> pd.Series:
    if not pd.api.types.is_numeric_dtype(s):
        raise ValueError(f"La serie debe ser numerica, recibio {s.dtype}")
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    low = q1 - k * iqr
    high = q3 + k * iqr
    return ((s < low) | (s > high)).fillna(False)


def resumen_outliers_iqr(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    filas = []
    for col in columnas:
        if col not in df.columns or not pd.api.types.is_numeric_dtype(df[col]):
            continue
        mascara = detectar_outliers_iqr(df[col])
        filas.append(
            {
                "columna": col,
                "outliers": int(mascara.sum()),
                "pct_outliers": round(float(mascara.mean() * 100), 2),
                "min": df[col].min(),
                "mediana": df[col].median(),
                "max": df[col].max(),
            }
        )
    return pd.DataFrame(filas).sort_values("pct_outliers", ascending=False)


def chi_square(df: pd.DataFrame, col_a: str, col_b: str) -> dict:
    sub = df[[col_a, col_b]].dropna()
    observed = pd.crosstab(sub[col_a], sub[col_b])
    chi2, p, dof, expected = stats.chi2_contingency(observed.values)
    expected_df = pd.DataFrame(expected, index=observed.index, columns=observed.columns)
    return {
        "chi2": float(chi2),
        "p_value": float(p),
        "dof": int(dof),
        "observed": observed,
        "expected": expected_df,
    }


def distancia_matriz(df: pd.DataFrame, metrica: str = "euclidean") -> np.ndarray:
    no_numericas = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if no_numericas:
        raise ValueError(f"Solo se aceptan columnas numericas. No numericas: {no_numericas}")
    return pairwise_distances(df.values, metric=metrica)


## 2. Carga de datos crudos

Los CSV descargados desde NASA tienen lineas de comentario al inicio. Por eso se usa `comment="#"`.


In [ ]:
CANDIDATE_DATA_DIRS = [Path("data"), Path("mineria") / "data"]
DATA_DIR = next(
    (
        data_dir
        for data_dir in CANDIDATE_DATA_DIRS
        if (data_dir / "cumulative_2026.06.01_20.09.17.csv").exists()
        and (data_dir / "PSCompPars_2026.06.01_20.09.10.csv").exists()
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError("No se encontraron los CSV. Deben estar en data/ o mineria/data/.")

KEPLER_PATH = DATA_DIR / "cumulative_2026.06.01_20.09.17.csv"
PSCOMP_PATH = DATA_DIR / "PSCompPars_2026.06.01_20.09.10.csv"

kepler_raw = pd.read_csv(KEPLER_PATH, comment="#")
pscomppars_raw = pd.read_csv(PSCOMP_PATH, comment="#")

kepler = limpiar_espacios_blanco(kepler_raw)
pscomppars = limpiar_espacios_blanco(pscomppars_raw)

print(f"Datos cargados desde: {DATA_DIR}")
pd.DataFrame(
    [
        resumen_dataframe("Kepler KOI cumulative", kepler),
        resumen_dataframe("PSCompPars", pscomppars),
    ]
)


In [ ]:
print("Kepler KOI")
display(kepler.head(5))

print("PSCompPars")
display(pscomppars.head(5))


## 3. Estructura general: df.info() y df.describe()

`info()` muestra tipos de pandas y nulos. `describe()` resume las variables numericas y ayuda a detectar escalas raras, colas largas y posibles outliers.


In [ ]:
print("INFO - Kepler KOI")
print(info_texto(kepler))

print("INFO - PSCompPars")
print(info_texto(pscomppars))


In [ ]:
print("DESCRIBE - Kepler KOI")
display(kepler.describe(include="all").T)

print("DESCRIBE - PSCompPars")
display(pscomppars.describe(include="all").T)


## 4. Clasificacion de atributos

Se clasifican las columnas con los cuatro tipos de clase: nominal, binario, ordinal y numerico. La columna `uso` no es un tipo nuevo; solo ayuda a recordar si una columna se conserva, si es objetivo o si se excluye por fuga de datos.


In [ ]:
KEPLER_HINTS = {
    "kepid": "nominal",
    "kepoi_name": "nominal",
    "kepler_name": "nominal",
    "koi_disposition": "nominal",
    "koi_pdisposition": "nominal",
    "koi_score": "numerico",
    "koi_fpflag_nt": "binario",
    "koi_fpflag_ss": "binario",
    "koi_fpflag_co": "binario",
    "koi_fpflag_ec": "binario",
    "koi_tce_plnt_num": "ordinal",
}

KEPLER_USO = {
    "kepid": "identificador",
    "kepoi_name": "identificador",
    "kepler_name": "identificador",
    "koi_disposition": "objetivo_clasificacion",
    "koi_prad": "objetivo_regresion",
    "koi_pdisposition": "excluir_fuga",
    "koi_score": "excluir_fuga",
    "koi_fpflag_nt": "excluir_fuga",
    "koi_fpflag_ss": "excluir_fuga",
    "koi_fpflag_co": "excluir_fuga",
    "koi_fpflag_ec": "excluir_fuga",
}

clasificacion_kepler = clasificar_atributos(kepler, hints=KEPLER_HINTS, uso=KEPLER_USO)
clasificacion_kepler


In [ ]:
PSCOMP_HINTS = {
    "pl_name": "nominal",
    "hostname": "nominal",
    "discoverymethod": "nominal",
    "disc_facility": "nominal",
    "disc_year": "ordinal",
    "pl_controv_flag": "binario",
    "ttv_flag": "binario",
    "sy_snum": "ordinal",
    "sy_pnum": "ordinal",
    "rastr": "nominal",
    "decstr": "nominal",
    "st_spectype": "nominal",
    "pl_bmassprov": "nominal",
    "st_metratio": "nominal",
}

PSCOMP_USO = {
    "pl_name": "identificador",
    "hostname": "identificador",
    "discoverymethod": "analisis",
    "disc_facility": "analisis",
    "disc_year": "analisis",
    "pl_rade": "analisis",
    "pl_orbper": "analisis",
    "st_teff": "analisis",
    "st_rad": "analisis",
    "st_mass": "analisis",
}

clasificacion_pscomppars = clasificar_atributos(pscomppars, hints=PSCOMP_HINTS, uso=PSCOMP_USO)
clasificacion_pscomppars


## 5. EDA de datos crudos

Aqui se revisa calidad y estructura: nulos, distribucion de clases, resumen numerico, matriz de correlacion, outliers y relaciones iniciales. Esto todavia no entrena modelos.


In [ ]:
print("Kepler - columnas con mas nulos")
display(columnas_con_mas_nulos(kepler, n=12))

print("PSCompPars - columnas con mas nulos")
display(columnas_con_mas_nulos(pscomppars, n=12))


In [ ]:
distribucion_clases = (
    kepler["koi_disposition"]
    .value_counts(dropna=False)
    .rename_axis("clase")
    .reset_index(name="n")
)
distribucion_clases["pct"] = (distribucion_clases["n"] / distribucion_clases["n"].sum() * 100).round(2)

display(distribucion_clases)

fig = px.bar(
    distribucion_clases,
    x="clase",
    y="n",
    text="pct",
    title="Distribucion de koi_disposition",
    template=PLOTLY_TEMPLATE,
    color="clase",
)
fig.update_layout(font=PLOTLY_FONT, showlegend=False)
fig.show()


In [ ]:
print("Metodos de descubrimiento en PSCompPars")
display(
    pscomppars["discoverymethod"]
    .value_counts(dropna=False)
    .head(12)
    .rename_axis("metodo")
    .reset_index(name="n")
)


In [ ]:
columnas_numericas_kepler = [
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_prad",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "ra",
    "dec",
    "koi_kepmag",
]

resumen_numerico(kepler, columnas_numericas_kepler)


In [ ]:
correlacion_kepler = kepler[columnas_numericas_kepler].corr(numeric_only=True)
correlacion_kepler


In [ ]:
fig = px.imshow(
    correlacion_kepler,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Matriz de correlacion - columnas numericas Kepler",
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(font=PLOTLY_FONT)
fig.show()


In [ ]:
resumen_outliers_iqr(kepler, columnas_numericas_kepler)


In [ ]:
fig = px.histogram(
    kepler,
    x="koi_prad",
    nbins=80,
    title="Distribucion del radio planetario (koi_prad)",
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(font=PLOTLY_FONT)
fig.show()

fig = px.histogram(
    kepler.assign(koi_prad_log=np.log1p(kepler["koi_prad"])),
    x="koi_prad_log",
    nbins=80,
    title="Radio planetario transformado con log1p",
    template=PLOTLY_TEMPLATE,
)
fig.update_layout(font=PLOTLY_FONT)
fig.show()


In [ ]:
variables_importantes = ["koi_period", "koi_depth", "koi_model_snr", "koi_steff"]

for col in variables_importantes:
    fig = px.histogram(
        kepler,
        x=col,
        color="koi_disposition",
        nbins=60,
        title=f"Distribucion de {col} por clase",
        template=PLOTLY_TEMPLATE,
    )
    fig.update_layout(font=PLOTLY_FONT)
    fig.show()


In [ ]:
correlaciones = (
    kepler[columnas_numericas_kepler]
    .corr(numeric_only=True)["koi_prad"]
    .drop("koi_prad")
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .reset_index()
)
correlaciones.columns = ["variable", "correlacion_con_koi_prad"]
correlaciones


## 6. Decisiones de preprocesamiento

Con base en el EDA, estas son las decisiones antes de modelar:

1. Quitar espacios en blanco en columnas nominales y convertir cadenas vacias a NaN.
2. Conservar identificadores solo para trazabilidad; no son atributos predictores.
3. Usar `koi_disposition` como objetivo de clasificacion.
4. Usar `koi_prad` como objetivo inicial de regresion y preparar `log1p(koi_prad)` por la cola extrema.
5. Excluir columnas con fuga de datos: `koi_score`, `koi_pdisposition` y `koi_fpflag_*`.
6. Imputar nulos y escalar dentro de un `Pipeline`, ajustado solo con train.
7. Usar one-hot encoding solo si se incluyen variables nominales predictoras. En Kepler se excluyen identificadores y columnas con fuga, por eso el primer modelo queda con variables numericas.


In [ ]:
columnas_fuga = [
    "koi_score",
    "koi_pdisposition",
    "koi_fpflag_nt",
    "koi_fpflag_ss",
    "koi_fpflag_co",
    "koi_fpflag_ec",
]

columnas_identificacion = ["kepid", "kepoi_name", "kepler_name"]
objetivo_clasificacion = "koi_disposition"
objetivo_regresion = "koi_prad"

features_numericas = [
    "koi_period",
    "koi_impact",
    "koi_duration",
    "koi_depth",
    "koi_teq",
    "koi_insol",
    "koi_model_snr",
    "koi_steff",
    "koi_slogg",
    "koi_srad",
    "ra",
    "dec",
    "koi_kepmag",
]

features_nominales = []

pd.DataFrame(
    {
        "grupo": [
            "identificadores_no_predictores",
            "objetivo_clasificacion",
            "objetivo_regresion",
            "features_numericas",
            "features_nominales",
            "excluir_por_fuga",
        ],
        "columnas": [
            ", ".join(columnas_identificacion),
            objetivo_clasificacion,
            objetivo_regresion,
            ", ".join(features_numericas),
            "ninguna por ahora: las nominales disponibles son ids o fuga",
            ", ".join(columnas_fuga),
        ],
    }
)


In [ ]:
transformadores = [
    (
        "numericas",
        Pipeline(
            steps=[
                ("imputar_mediana", SimpleImputer(strategy="median")),
                ("escalar", StandardScaler()),
            ]
        ),
        features_numericas,
    )
]

if features_nominales:
    transformadores.append(
        (
            "nominales",
            Pipeline(
                steps=[
                    ("imputar_desconocido", SimpleImputer(strategy="constant", fill_value="Desconocido")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            features_nominales,
        )
    )

preprocesador = ColumnTransformer(transformadores, remainder="drop")
preprocesador


## 7. Separacion train/test antes de ajustar transformaciones

Esta parte es critica: el imputador y el scaler se ajustan con `X_train`, no con todo el dataset. Asi se evita fuga de datos.


In [ ]:
kepler_modelo_clasificacion = kepler[
    columnas_identificacion
    + [objetivo_clasificacion, objetivo_regresion]
    + features_numericas
].dropna(subset=[objetivo_clasificacion]).copy()

X_clf = kepler_modelo_clasificacion[features_numericas + features_nominales]
y_clf = kepler_modelo_clasificacion[objetivo_clasificacion]

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.20,
    random_state=42,
    stratify=y_clf,
)

X_train_clf_preparado = preprocesador.fit_transform(X_train_clf)
X_test_clf_preparado = preprocesador.transform(X_test_clf)

pd.DataFrame(
    {
        "particion": ["X_train", "X_test", "y_train", "y_test"],
        "shape": [
            X_train_clf.shape,
            X_test_clf.shape,
            y_train_clf.shape,
            y_test_clf.shape,
        ],
    }
)


In [ ]:
kepler_modelo_regresion = kepler[
    columnas_identificacion
    + [objetivo_regresion]
    + features_numericas
].dropna(subset=[objetivo_regresion]).copy()

X_reg = kepler_modelo_regresion[features_numericas + features_nominales]
y_reg = np.log1p(kepler_modelo_regresion[objetivo_regresion])

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.20,
    random_state=42,
)

preprocesador_regresion = ColumnTransformer(transformadores, remainder="drop")
X_train_reg_preparado = preprocesador_regresion.fit_transform(X_train_reg)
X_test_reg_preparado = preprocesador_regresion.transform(X_test_reg)

pd.DataFrame(
    {
        "particion": ["X_train_reg", "X_test_reg", "y_train_reg", "y_test_reg"],
        "shape": [
            X_train_reg.shape,
            X_test_reg.shape,
            y_train_reg.shape,
            y_test_reg.shape,
        ],
    }
)


In [ ]:
pscomppars_preprocesado = pscomppars[
    [
        "pl_name",
        "hostname",
        "discoverymethod",
        "disc_year",
        "disc_facility",
        "pl_orbper",
        "pl_rade",
        "pl_bmasse",
        "pl_eqt",
        "st_teff",
        "st_rad",
        "st_mass",
        "sy_dist",
        "ra",
        "dec",
    ]
].copy()

# Esta tabla se usara principalmente como referencia para dashboard/warehouse, no para entrenar el primer modelo.
for col in ["pl_name", "hostname", "discoverymethod", "disc_facility"]:
    pscomppars_preprocesado[col] = pscomppars_preprocesado[col].fillna("Desconocido")

print("Kepler clasificacion listo:", X_train_clf_preparado.shape, X_test_clf_preparado.shape)
print("Kepler regresion listo:", X_train_reg_preparado.shape, X_test_reg_preparado.shape)
print("PSCompPars referencia:", pscomppars_preprocesado.shape)

display(kepler_modelo_clasificacion.head())
display(pscomppars_preprocesado.head())
